# MACD Signal Accuracy Analysis
How accurate is MACD as a trend signal? Testing across different assets and timeframes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

plt.style.use('dark_background')

def compute_macd(prices, fast=12, slow=26, signal=9):
    ema_fast = prices.ewm(span=fast).mean()
    ema_slow = prices.ewm(span=slow).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal).mean()
    histogram = macd - macd_signal
    return macd, macd_signal, histogram

def macd_backtest(ticker, period='5y'):
    df = yf.download(ticker, period=period, progress=False)
    macd, signal, hist = compute_macd(df['Close'])
    
    # Signal: buy on MACD cross above signal, sell on cross below
    position = 0
    capital = 100_000
    cash = capital
    shares = 0
    trades = []
    
    for i in range(1, len(df)):
        price = df['Close'].iloc[i]
        cross_up = hist.iloc[i] > 0 and hist.iloc[i-1] <= 0
        cross_down = hist.iloc[i] < 0 and hist.iloc[i-1] >= 0
        
        if cross_up and shares == 0:
            shares = int(cash * 0.95 / price)
            entry = price
            cash -= shares * price
        elif cross_down and shares > 0:
            pnl = (price - entry) * shares
            trades.append({'pnl': pnl, 'ret_pct': (price/entry-1)*100})
            cash += shares * price
            shares = 0
    
    final = cash + shares * df['Close'].iloc[-1]
    wins = [t for t in trades if t['pnl'] > 0]
    
    return {
        'ticker': ticker,
        'total_return': f'{(final/capital-1)*100:.1f}%',
        'trades': len(trades),
        'win_rate': f'{len(wins)/max(len(trades),1)*100:.0f}%',
        'avg_win': f'{np.mean([t["ret_pct"] for t in wins]):.1f}%' if wins else '-',
        'avg_loss': f'{np.mean([t["ret_pct"] for t in trades if t["pnl"]<0]):.1f}%' if len(trades)>len(wins) else '-',
    }

In [ ]:
tickers = ['SPY', 'QQQ', 'AAPL', 'MSFT', 'TSLA', 'NVDA', 'GLD', 'TLT']
results = [macd_backtest(t) for t in tickers]
print(pd.DataFrame(results).to_string(index=False))

In [ ]:
# Detailed plot for SPY
df = yf.download('SPY', period='2y', progress=False)
macd, signal, hist = compute_macd(df['Close'])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), height_ratios=[2, 1], sharex=True)

ax1.plot(df.index, df['Close'], color='white', linewidth=0.8)
ax1.set_title('SPY + MACD Signals', fontsize=14)
ax1.grid(alpha=0.2)

# Mark buy/sell signals
for i in range(1, len(df)):
    if hist.iloc[i] > 0 and hist.iloc[i-1] <= 0:
        ax1.axvline(df.index[i], color='#34d399', alpha=0.3, linewidth=0.5)
    elif hist.iloc[i] < 0 and hist.iloc[i-1] >= 0:
        ax1.axvline(df.index[i], color='#ef4444', alpha=0.3, linewidth=0.5)

colors = ['#34d399' if v > 0 else '#ef4444' for v in hist]
ax2.bar(df.index, hist, color=colors, width=1, alpha=0.7)
ax2.plot(df.index, macd, color='#22d3ee', linewidth=0.8, label='MACD')
ax2.plot(df.index, signal, color='#f97316', linewidth=0.8, label='Signal')
ax2.legend(fontsize=9)
ax2.grid(alpha=0.2)

plt.tight_layout()
plt.show()